# 02 — MAESTRO post-training analysis

This notebook demonstrates downstream analyses you can perform after training a MAESTRO model on CyTOF data. It covers:

1. **Loading a trained model and data**
2. **Reconstruction visualization** — masking cells and comparing reconstructions
3. **Latent embedding extraction** — generating patient-level representations
4. **UMAP visualization** — embedding space colored by metadata (diagnosis, sex, age)
5. **Diagnosis classification** — logistic regression on latent embeddings
6. **Sex prediction** — binary classification from embeddings
7. **Age regression** — Ridge regression from embeddings
8. **Cell type proportion prediction** — predicting gated proportions from embeddings
9. **Batch effect analysis** — measuring robustness to marker intensity shifts
10. **Pool attention interpretation** — understanding which cell types the model attends to

This executed analysis targets the completed `maestro-allof-prepro-7407312` production run. It uses the best training-loss checkpoint, the original pre-cyDM `allof` and `prepro` samples, and patient-grouped cross-validation for exploratory downstream probes. Because the encoder was pretrained on every sample, these probes measure information present in the representation; they are not held-out estimates of foundation-model transfer.

## 0. Setup & Imports

In [1]:
import os
import re
import sys
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import umap
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    mean_absolute_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm

# Support launching Jupyter from either the repository root or notebooks/.
project_root = Path.cwd().resolve()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
from src.data.dataset import CyTOFDataset
from src.models.maestro import MAESTRO, ruler_masking

warnings.filterwarnings("ignore", category=FutureWarning)

plt.style.use("default")
plt.rcParams.update(
    {
        "figure.figsize": (5, 4),
        "font.family": "sans-serif",
        "figure.dpi": 150,
        "axes.grid": False,
    }
)

random_seed = 206
np.random.seed(random_seed)
torch.manual_seed(random_seed)

print("Imports complete.")

[KeOps] Compiling cuda jit compiler engine ... 

OK


[pyKeOps] Compiling nvrtc binder for python ... 

OK


Imports complete.


## 1. Load Data & Model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.set_float32_matmul_precision("high")
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is unavailable; production analysis will be slow.")

Using device: NVIDIA B200


### 1a. Load the CyTOF dataset

The analysis uses the same two pre-cyDM source directories as training. Each sample is represented by a reproducible 100,000-cell view; smaller samples are sampled with replacement by `CyTOFDataset`, matching the training data layer.

In [3]:
source_root = project_root / "data/raw/preprocessed-cyto-sources"
data_directories = [source_root / "allof", source_root / "prepro"]
dataset = CyTOFDataset(data_dirs=data_directories, subset_size=100_000)

print(f"Number of samples: {len(dataset)}")
print(f"Shared markers ({len(dataset.shared_markers)}): {dataset.shared_markers}")

[Panel] allof: 30 markers, 1 renamed
[Panel]   CD8a -> CD8
[Panel] prepro: 30 markers, 1 renamed
[Panel]   CD8a -> CD8
[Panel] Shared markers (30): ['CD11c', 'CD123', 'CD127', 'CD14', 'CD16', 'CD161', 'CD183', 'CD185', 'CD19', 'CD194', 'CD196', 'CD197', 'CD20', 'CD25', 'CD27', 'CD28', 'CD294', 'CD3', 'CD38', 'CD4', 'CD45', 'CD45RA', 'CD45RO', 'CD56', 'CD57', 'CD66b', 'CD8', 'HLA-DR', 'IgD', 'TCRgd']
Initialized dataset: 1870 samples from 2 training directories
Shared markers (30): ['CD11c', 'CD123', 'CD127', 'CD14', 'CD16', 'CD161', 'CD183', 'CD185', 'CD19', 'CD194', 'CD196', 'CD197', 'CD20', 'CD25', 'CD27', 'CD28', 'CD294', 'CD3', 'CD38', 'CD4', 'CD45', 'CD45RA', 'CD45RO', 'CD56', 'CD57', 'CD66b', 'CD8', 'HLA-DR', 'IgD', 'TCRgd']
Number of samples: 1870
Shared markers (30): ['CD11c', 'CD123', 'CD127', 'CD14', 'CD16', 'CD161', 'CD183', 'CD185', 'CD19', 'CD194', 'CD196', 'CD197', 'CD20', 'CD25', 'CD27', 'CD28', 'CD294', 'CD3', 'CD38', 'CD4', 'CD45', 'CD45RA', 'CD45RO', 'CD56', 'CD57', '

In [4]:
sample_tensors = []
sample_cell_types = []
sample_names = []

for sample_index in tqdm(range(len(dataset)), desc="Loading samples"):
    sample_tensor, cell_types, sample_name = dataset[sample_index]
    sample_tensors.append(sample_tensor.unsqueeze(0))
    sample_cell_types.append(cell_types)
    sample_names.append(sample_name)

print(f"Loaded {len(sample_tensors)} samples.")
print(f"Example sample shape: {sample_tensors[0].shape}")
print(f"Discovered cell types: {dataset.get_all_cell_types()}")

Loading samples:   0%|          | 0/1870 [00:00<?, ?it/s]

Loaded 1870 samples.
Example sample shape: torch.Size([1, 100000, 30])
Discovered cell types: ['B cell', 'Basophil', 'Eosinophil', 'Monocyte Classical', 'Monocyte Nonclassical', 'NK cell', 'Neutrophil', 'Plasmablast', 'T cell CD4 EMRA', 'T cell CD4 Mem', 'T cell CD4 Naive', 'T cell CD8 Mem', 'T cell CD8 Naive', 'T cell DN', 'T cell gd', 'mDC', 'pDC']


### 1b. Load metadata

In [5]:
metadata = pd.read_csv(project_root / "data/metadata/allof/all-metadata.csv")
metadata["sample_name"] = metadata["filename"].map(
    lambda filename: Path(str(filename)).stem
)
metadata["analysis_diagnosis"] = (
    metadata["diagnosis"]
    .astype("string")
    .str.replace(r"^BatchControl.*", "Batch Control", regex=True)
    .str.replace("Term infant", "Term Infant")
)
metadata["group_id"] = metadata["patient_id"].astype("string")
missing_patient = metadata["group_id"].isna()
metadata.loc[missing_patient, "group_id"] = (
    "sample::" + metadata.loc[missing_patient, "sample_name"]
)

matched_metadata = metadata[metadata["sample_name"].isin(sample_names)]
assert len(matched_metadata) == len(sample_names)
assert matched_metadata["sample_name"].is_unique
print(f"Metadata rows matched: {len(matched_metadata):,}/{len(sample_names):,}")
print(f"Unique patients/groups: {matched_metadata['group_id'].nunique():,}")
matched_metadata.head()

Metadata rows matched: 1,870/1,870
Unique patients/groups: 955


,filename,dataset,platform,diagnosis,treatment,patient_id,sex,age,timepoint,race,...,pfs_event_flag,os_time_months,os_event_flag,QC,irAE_grade,previous_autoimmune_dx,immunosupressant_therapy,sample_name,analysis_diagnosis,group_id
0,Heparin MDIPA_DORA_HD_05Nov24,allof,CyTOF,BatchControlHD2 Draw 2,NaN,NaN,Male,33.0,NaN,Asian,...,NaN,NaN,NaN,PASS,NaN,NaN,NaN,Heparin MDIPA_DORA_HD_05Nov24,Batch Control,sample::Heparin MDIPA_DORA_HD_05Nov24
1,MDIPA_845061_097_Blood Draw_PBMC,allof,CyTOF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MDIPA_845061_097_Blood Draw_PBMC,<NA>,sample::MDIPA_845061_097_Blood Draw_PBMC
2,MDIPA_845061_097_Blood Draw_WBHep,allof,CyTOF,Healthy,NaN,845061.097,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,TBD,NaN,NaN,NaN,MDIPA_845061_097_Blood Draw_WBHep,Healthy,845061.097
3,MDIPA_845061_098_Blood Draw_PBMC,allof,CyTOF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MDIPA_845061_098_Blood Draw_PBMC,<NA>,sample::MDIPA_845061_098_Blood Draw_PBMC
4,MDIPA_845061_098_Blood Draw_WBHep,allof,CyTOF,Healthy,NaN,845061.098,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,TBD,NaN,NaN,NaN,MDIPA_845061_098_Blood Draw_WBHep,Healthy,845061.098


### 1c. Load a trained MAESTRO model

The best checkpoint is selected by the training-loss callback. `last.ckpt` is retained for resuming optimization, while the best checkpoint is the appropriate default for inference.

In [6]:
run_name = os.environ.get("MAESTRO_ANALYSIS_RUN", "maestro-allof-prepro-7407312")
run_directory = project_root / "experiments" / run_name
analysis_directory = run_directory / "analysis"
analysis_directory.mkdir(parents=True, exist_ok=True)
configuration_path = run_directory / "config.pth"
best_checkpoint_directories = sorted(run_directory.glob("best-epoch=*.ckpt"))
assert len(best_checkpoint_directories) == 1
checkpoint_path = (
    best_checkpoint_directories[0] / "checkpoint" / "mp_rank_00_model_states.pt"
)

model_configuration = torch.load(
    configuration_path, map_location="cpu", weights_only=False
)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
state_dict = {
    state_key.removeprefix("model."): state_value
    for state_key, state_value in checkpoint["module"].items()
}

model = MAESTRO(
    dim_input=model_configuration["dim_input"],
    dim_output=model_configuration["dim_output"],
    num_inds=model_configuration["num_inds"],
    dim_hidden=model_configuration["dim_hidden"],
    dim_latent=model_configuration["dim_latent"],
    num_heads=model_configuration["num_heads"],
    num_outputs=model_configuration["num_outputs"],
    ln=model_configuration["ln"],
    number_cells_subset=model_configuration["number_cells_subset"],
    student_temperature=model_configuration["student_temperature"],
    teacher_temperature=model_configuration["teacher_temperature"],
    teacher_temperature_step=model_configuration.get("teacher_temperature_step", 0.01),
    sinkhorn_start=model_configuration.get("sinkhorn_start", 0),
    center_momentum=model_configuration.get("center_momentum", 0.9),
    teacher_beta=model_configuration.get("teacher_beta", 0.99),
).to(device)
model.load_state_dict(state_dict, strict=True)
model.eval()
model_dtype = next(model.parameters()).dtype
print(f"Loaded {best_checkpoint_directories[0].name} as {model_dtype}.")
print(f"Analysis outputs: {analysis_directory}")

Loaded best-epoch=498.ckpt as torch.float32.
Analysis outputs: /vast/projects/arjunraj/rohitproject/maestro/experiments/maestro-allof-prepro-7407312/analysis


---
## 2. Training health and GPU telemetry

The objective changes from energy distance to Sinkhorn at epoch 25, so the loss discontinuity at that boundary is expected. The two phases should be interpreted separately.

In [7]:
training_log = (run_directory / "job/slurm.log").read_text(errors="replace")
epoch_pattern = re.compile(
    r"Epoch (\d+) \[(energy|sinkhorn)\].*?"
    r"Duration: ([0-9.]+) min.*?Loss: ([0-9.eE+-]+).*?"
    r"Recon: ([0-9.eE+-]+).*?Distillation: ([0-9.eE+-]+)"
)
epoch_records = epoch_pattern.findall(training_log)
training_history = pd.DataFrame(
    [
        {
            "epoch": int(epoch),
            "objective": objective,
            "duration_minutes": float(duration),
            "total_loss": float(total_loss),
            "reconstruction_loss": float(reconstruction_loss),
            "distillation_loss": float(distillation_loss),
        }
        for (
            epoch,
            objective,
            duration,
            total_loss,
            reconstruction_loss,
            distillation_loss,
        ) in epoch_records
    ]
)
assert len(training_history) == model_configuration["epochs"]
training_history.to_csv(analysis_directory / "training_history.csv", index=False)

_, training_axes = plt.subplots(1, 2, figsize=(13, 4))
for objective, objective_frame in training_history.groupby("objective"):
    training_axes[0].plot(
        objective_frame["epoch"],
        objective_frame["total_loss"],
        label=f"{objective} total",
    )
training_axes[0].plot(
    training_history["epoch"],
    training_history["distillation_loss"],
    label="distillation",
    alpha=0.8,
)
training_axes[0].set_yscale("log")
training_axes[0].set_xlabel("Epoch")
training_axes[0].set_ylabel("Loss (log scale)")
training_axes[0].set_title("Training objectives")
training_axes[0].legend(fontsize=8)
training_axes[1].plot(training_history["epoch"], training_history["duration_minutes"])
training_axes[1].axvline(25, color="red", linestyle="--", alpha=0.6)
training_axes[1].set_xlabel("Epoch")
training_axes[1].set_ylabel("Minutes")
training_axes[1].set_title("Epoch duration")
plt.tight_layout()
plt.savefig(analysis_directory / "training_health.png", bbox_inches="tight")
plt.show()

training_history.groupby("objective").agg(
    epochs=("epoch", "size"),
    mean_minutes=("duration_minutes", "mean"),
    first_loss=("total_loss", "first"),
    final_loss=("total_loss", "last"),
    minimum_loss=("total_loss", "min"),
)

,epochs,mean_minutes,first_loss,final_loss,minimum_loss
objective,,,,,
energy,25,1.588000,5.018115,0.409325,0.390029
sinkhorn,475,6.621684,8.244436,2.132582,2.092541


In [8]:
gpu_frame = pd.read_csv(run_directory / "job/gpu.csv")
gpu_summary = (
    gpu_frame.groupby("gpu_index")
    .agg(
        mean_utilization_percent=("utilization_gpu_percent", "mean"),
        median_utilization_percent=("utilization_gpu_percent", "median"),
        maximum_memory_mib=("memory_used_mib", "max"),
        mean_power_watts=("power_draw_watts", "mean"),
        maximum_temperature_c=("temperature_c", "max"),
    )
    .reset_index()
)
gpu_summary.to_csv(analysis_directory / "gpu_summary.csv", index=False)
gpu_summary

,gpu_index,mean_utilization_percent,median_utilization_percent,maximum_memory_mib,mean_power_watts,maximum_temperature_c
0,0,95.547513,98.0,79090,432.680235,46
1,1,96.758294,98.0,73814,426.399501,46
2,2,96.788133,98.0,73810,438.989096,55
3,3,97.548995,99.0,73818,337.715357,50


---
## 3. Reconstruction Visualization

Mask a portion of cells and compare the model's reconstruction to the original data using UMAP.

In [9]:
cell_type_colors = {
    "Neutrophil": "#0F4C81",
    "T cell CD4 Mem": "#FF6F61",
    "T cell CD4 Naive": "#009874",
    "T cell CD8 Mem": "#F5DF4D",
    "T cell CD8 Naive": "#BF1932",
    "B cell": "#92A8D1",
    "Monocyte Classical": "#88B04B",
    "Monocyte Nonclassical": "#9B1B30",
    "NK cell": "#E2583E",
    "T cell CD4 EMRA": "#5A5B9F",
    "mDC": "#F0C05A",
    "pDC": "#53B0AE",
    "Eosinophil": "#F7CAC9",
    "Basophil": "#5F4B8B",
}
default_color = "#AAAAAA"

number_visualized_samples = 3
visualization_sample_indices = np.random.choice(
    len(sample_tensors), number_visualized_samples, replace=False
)
mask_ratio = 0.5

_, subplot_axes = plt.subplots(
    number_visualized_samples,
    4,
    figsize=(16, 4 * number_visualized_samples),
)
column_titles = [
    "Cell Types (original)",
    "Unmasked",
    "Masked (target)",
    "Predicted",
]

for row_index, sample_index in enumerate(visualization_sample_indices):
    with torch.no_grad():
        sample_tensor = sample_tensors[sample_index].to(
            device=device, dtype=model_dtype
        )
        cell_type_indices = sample_cell_types[sample_index]
        sample_name = sample_names[sample_index]

        number_cells = sample_tensor.shape[1]
        visualization_size = min(5000, number_cells)
        selected_cell_indices = torch.randperm(number_cells)[:visualization_size]
        sampled_input = sample_tensor[:, selected_cell_indices.to(device), :]
        sampled_cell_type_indices = cell_type_indices[selected_cell_indices]

        mask = ruler_masking(sampled_input, mask_ratio)
        batch_size, _, marker_count = sampled_input.shape
        unmasked_cells = sampled_input[~mask].view(batch_size, -1, marker_count)
        masked_target_cells = sampled_input[mask].view(batch_size, -1, marker_count)

        encoded_cells, *_ = model.student.forward_encoder(unmasked_cells)
        sample_embedding, _, _ = model.student.forward_pooling(encoded_cells)
        reconstructed_cells = model.student.forward_decoder(sample_embedding, mask)

        number_masked_cells = masked_target_cells.shape[1]
        prediction_indices = torch.randperm(
            reconstructed_cells.shape[1], device=device
        )[:number_masked_cells]
        sampled_predictions = reconstructed_cells[:, prediction_indices, :]

        unmasked_values = unmasked_cells.squeeze(0).float().cpu().numpy()
        target_values = masked_target_cells.squeeze(0).float().cpu().numpy()
        prediction_values = sampled_predictions.squeeze(0).float().cpu().numpy()

    combined_values = np.vstack([unmasked_values, target_values, prediction_values])
    umap_reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42)
    embedding = umap_reducer.fit_transform(combined_values)

    number_unmasked = len(unmasked_values)
    number_targets = len(target_values)
    unmasked_embedding = embedding[:number_unmasked]
    target_embedding = embedding[number_unmasked : number_unmasked + number_targets]
    prediction_embedding = embedding[number_unmasked + number_targets :]

    cpu_mask = mask.squeeze(0).cpu()
    unmasked_indices = (~cpu_mask).nonzero(as_tuple=True)[0]
    masked_indices = cpu_mask.nonzero(as_tuple=True)[0]
    unmasked_cell_type_names = [
        dataset.get_cell_type_name(sampled_cell_type_indices[index].item())
        for index in unmasked_indices
    ]
    masked_cell_type_names = [
        dataset.get_cell_type_name(sampled_cell_type_indices[index].item())
        for index in masked_indices
    ]
    cell_type_names = unmasked_cell_type_names + masked_cell_type_names

    x_limits = [embedding[:, 0].min() - 0.5, embedding[:, 0].max() + 0.5]
    y_limits = [embedding[:, 1].min() - 0.5, embedding[:, 1].max() + 0.5]

    subplot_axis = subplot_axes[row_index, 0]
    point_colors = [
        cell_type_colors.get(cell_type, default_color) for cell_type in cell_type_names
    ]
    subplot_axis.scatter(
        np.vstack([unmasked_embedding, target_embedding])[:, 0],
        np.vstack([unmasked_embedding, target_embedding])[:, 1],
        c=point_colors,
        s=1,
        alpha=0.5,
    )
    subplot_axis.set_xlim(x_limits)
    subplot_axis.set_ylim(y_limits)
    subplot_axis.set_xticks([])
    subplot_axis.set_yticks([])
    subplot_axis.set_ylabel(sample_name, fontsize=9)
    if row_index == 0:
        subplot_axis.set_title(column_titles[0], fontsize=10)

    subplot_axis = subplot_axes[row_index, 1]
    subplot_axis.scatter(
        unmasked_embedding[:, 0],
        unmasked_embedding[:, 1],
        c="#FFB703",
        s=1,
        alpha=0.5,
    )
    subplot_axis.set_xlim(x_limits)
    subplot_axis.set_ylim(y_limits)
    subplot_axis.set_xticks([])
    subplot_axis.set_yticks([])
    if row_index == 0:
        subplot_axis.set_title(column_titles[1], fontsize=10)

    subplot_axis = subplot_axes[row_index, 2]
    subplot_axis.scatter(
        target_embedding[:, 0],
        target_embedding[:, 1],
        c="#E63946",
        s=1,
        alpha=0.5,
    )
    subplot_axis.set_xlim(x_limits)
    subplot_axis.set_ylim(y_limits)
    subplot_axis.set_xticks([])
    subplot_axis.set_yticks([])
    if row_index == 0:
        subplot_axis.set_title(column_titles[2], fontsize=10)

    subplot_axis = subplot_axes[row_index, 3]
    subplot_axis.scatter(
        prediction_embedding[:, 0],
        prediction_embedding[:, 1],
        c="#0077B6",
        s=1,
        alpha=0.5,
    )
    subplot_axis.set_xlim(x_limits)
    subplot_axis.set_ylim(y_limits)
    subplot_axis.set_xticks([])
    subplot_axis.set_yticks([])
    if row_index == 0:
        subplot_axis.set_title(column_titles[3], fontsize=10)

plt.suptitle(f"Reconstruction Visualization (mask ratio = {mask_ratio})", fontsize=13)
plt.tight_layout()
plt.savefig(analysis_directory / "reconstruction_umap.png", bbox_inches="tight")
plt.show()

/vast/projects/arjunraj/rohitproject/maestro/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/vast/projects/arjunraj/rohitproject/maestro/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/vast/projects/arjunraj/rohitproject/maestro/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


---
## 4. Latent Embedding Extraction

Generate a single latent vector per patient by passing all cells through the encoder and pooling layer.

In [10]:
assert model is not None, "Model not loaded."

sample_embeddings = {}

for sample_index in tqdm(
    range(len(sample_tensors)), desc="Generating latent embeddings"
):
    with torch.no_grad():
        sample_tensor = sample_tensors[sample_index].to(
            device=device, dtype=model_dtype
        )
        encoded_cells, *_ = model.student.forward_encoder(sample_tensor)
        sample_embedding, _, _ = model.student.forward_pooling(encoded_cells)
        sample_embeddings[sample_names[sample_index]] = sample_embedding.float().cpu()

embedding_values = {
    sample_name: embedding_tensor.flatten().numpy()
    for sample_name, embedding_tensor in sample_embeddings.items()
}

embedding_dimension = next(iter(embedding_values.values())).shape[0]
embedding_frame = pd.DataFrame.from_dict(embedding_values, orient="index")
embedding_frame.reset_index(inplace=True)
embedding_frame.columns = ["sample_name"] + [
    f"Feature_{feature_index + 1}" for feature_index in range(embedding_dimension)
]

embedding_frame.to_csv(analysis_directory / "maestro_embeddings.csv", index=False)
print(f"Latent DataFrame shape: {embedding_frame.shape}")
embedding_frame.head()

Generating latent embeddings:   0%|          | 0/1870 [00:00<?, ?it/s]

Latent DataFrame shape: (1870, 257)


,sample_name,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,...,Feature_247,Feature_248,Feature_249,Feature_250,Feature_251,Feature_252,Feature_253,Feature_254,Feature_255,Feature_256
0,Heparin MDIPA_DORA_HD_05Nov24,0.431149,0.110932,-0.096084,0.832422,0.073925,0.063371,-0.502685,0.241958,-0.214959,...,-0.025706,-0.184105,0.793049,0.111961,-0.134635,0.118675,-0.444425,-0.225766,0.401672,0.165089
1,MDIPA_845061_097_Blood Draw_PBMC,0.425565,0.063335,0.199516,-0.047457,0.319586,0.639610,-0.916214,1.471097,0.068251,...,-0.812649,-0.901550,-0.272593,-0.501907,-0.216630,0.262099,-0.025269,-0.236292,0.347343,-0.046232
2,MDIPA_845061_097_Blood Draw_WBHep,0.198634,0.070127,-0.162406,0.173945,0.694853,0.541741,0.354940,0.009088,-0.229381,...,-0.050626,-0.856886,-0.153890,-0.372920,-0.321517,0.312609,0.009177,0.134248,0.416542,0.161666
3,MDIPA_845061_098_Blood Draw_PBMC,-0.172502,-0.210564,-0.189274,-0.400540,0.528381,1.148862,-0.451843,0.602664,0.024231,...,-1.056723,-0.983199,0.236354,0.000330,0.089029,0.298712,-0.214089,-0.514524,0.559900,-0.000524
4,MDIPA_845061_098_Blood Draw_WBHep,0.049656,-0.027405,-0.365984,0.105141,0.561214,0.704941,0.179432,-0.267872,0.012164,...,0.124247,-0.747821,0.337862,0.165279,-0.451263,0.293257,-0.089290,-0.023845,0.176602,-0.024598


---
## 5. UMAP Visualization of Latent Embeddings

Visualize the patient-level latent space, colored by diagnosis, sex, and age.

In [11]:
analysis_frame = pd.merge(
    embedding_frame, matched_metadata, on="sample_name", how="inner"
)
print(f"Merged DataFrame shape: {analysis_frame.shape}")

feature_columns = [
    column for column in analysis_frame.columns if column.startswith("Feature_")
]
latent_matrix = analysis_frame[feature_columns].values

umap_reducer = umap.UMAP(random_state=42, n_neighbors=15, min_dist=0.5)
umap_embedding = umap_reducer.fit_transform(latent_matrix)
analysis_frame["UMAP1"] = umap_embedding[:, 0]
analysis_frame["UMAP2"] = umap_embedding[:, 1]
analysis_frame.to_csv(
    analysis_directory / "maestro_embeddings_with_metadata.csv", index=False
)

Merged DataFrame shape: (1870, 279)


/vast/projects/arjunraj/rohitproject/maestro/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [12]:
diagnosis_counts = analysis_frame["analysis_diagnosis"].value_counts()
display_diagnoses = diagnosis_counts.head(12).index.tolist()
diagnosis_colors = dict(
    zip(
        display_diagnoses,
        sns.color_palette("tab20", len(display_diagnoses)),
        strict=True,
    )
)

_, subplot_axes = plt.subplots(1, 3, figsize=(15, 4))

subplot_axis = subplot_axes[0]
for diagnosis, color in diagnosis_colors.items():
    diagnosis_mask = analysis_frame["analysis_diagnosis"] == diagnosis
    subplot_axis.scatter(
        analysis_frame.loc[diagnosis_mask, "UMAP1"],
        analysis_frame.loc[diagnosis_mask, "UMAP2"],
        c=color,
        label=diagnosis,
        s=15,
        alpha=0.7,
    )
subplot_axis.set_title("Colored by Diagnosis")
subplot_axis.legend(
    fontsize=6, markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left"
)
subplot_axis.set_xticks([])
subplot_axis.set_yticks([])
subplot_axis.set_xlabel("UMAP1")
subplot_axis.set_ylabel("UMAP2")

sex_colors = {"Male": "#88CCEE", "Female": "#CC6677"}
subplot_axis = subplot_axes[1]
for sex, color in sex_colors.items():
    sex_mask = analysis_frame["sex"] == sex
    subplot_axis.scatter(
        analysis_frame.loc[sex_mask, "UMAP1"],
        analysis_frame.loc[sex_mask, "UMAP2"],
        c=color,
        label=sex,
        s=15,
        alpha=0.7,
    )
subplot_axis.set_title("Colored by Sex")
subplot_axis.legend(fontsize=8, markerscale=2)
subplot_axis.set_xticks([])
subplot_axis.set_yticks([])
subplot_axis.set_xlabel("UMAP1")

subplot_axis = subplot_axes[2]
age_scatter = subplot_axis.scatter(
    analysis_frame["UMAP1"],
    analysis_frame["UMAP2"],
    c=pd.to_numeric(analysis_frame["age"], errors="coerce"),
    cmap="coolwarm",
    s=15,
    alpha=0.7,
)
plt.colorbar(age_scatter, ax=subplot_axis, label="Age")
subplot_axis.set_title("Colored by Age")
subplot_axis.set_xticks([])
subplot_axis.set_yticks([])
subplot_axis.set_xlabel("UMAP1")

plt.suptitle("MAESTRO Latent Embedding UMAP", fontsize=13)
plt.tight_layout()
plt.savefig(analysis_directory / "maestro_embedding_umap.png", bbox_inches="tight")
plt.show()

/tmp/ipykernel_4192881/4007350004.py:16: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  subplot_axis.scatter(
/tmp/ipykernel_4192881/4007350004.py:16: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  subplot_axis.scatter(
/tmp/ipykernel_4192881/4007350004.py:16: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with

---
## 6. Diagnosis Classification

Train a logistic regression classifier on the latent embeddings to predict diagnosis. Folds are stratified while keeping every patient in exactly one fold. Classes need at least 10 samples and 10 patient/groups.

In [13]:
diagnosis_support = analysis_frame.groupby("analysis_diagnosis").agg(
    samples=("sample_name", "size"), groups=("group_id", "nunique")
)
eligible_diagnoses = diagnosis_support.query("samples >= 10 and groups >= 10").index
diagnosis_frame = analysis_frame[
    analysis_frame["analysis_diagnosis"].isin(eligible_diagnoses)
].reset_index(drop=True)
features = diagnosis_frame[feature_columns].values
diagnosis_labels = diagnosis_frame["analysis_diagnosis"].to_numpy()
diagnosis_groups = diagnosis_frame["group_id"].to_numpy()

diagnosis_classes = sorted(np.unique(diagnosis_labels))
print(f"Classes: {diagnosis_classes}, N={len(diagnosis_labels)}")

stratified_folds = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=random_seed
)
observed_labels, predicted_labels = [], []
fold_accuracies = []

for fold_index, (training_indices, testing_indices) in enumerate(
    stratified_folds.split(features, diagnosis_labels, groups=diagnosis_groups)
):
    feature_scaler = StandardScaler()
    training_features = feature_scaler.fit_transform(features[training_indices])
    testing_features = feature_scaler.transform(features[testing_indices])
    training_labels = diagnosis_labels[training_indices]
    testing_labels = diagnosis_labels[testing_indices]

    classifier = LogisticRegression(
        max_iter=2_000, class_weight="balanced", random_state=random_seed
    )
    classifier.fit(training_features, training_labels)
    fold_predictions = classifier.predict(testing_features)

    fold_accuracy = accuracy_score(testing_labels, fold_predictions)
    fold_accuracies.append(fold_accuracy)
    observed_labels.extend(testing_labels)
    predicted_labels.extend(fold_predictions)
    print(f"  Fold {fold_index + 1}: Accuracy = {fold_accuracy:.3f}")

print(
    f"\nMean Accuracy: {np.mean(fold_accuracies):.3f} +/- {np.std(fold_accuracies):.3f}"
)
diagnosis_prediction_frame = pd.DataFrame(
    {"observed": observed_labels, "predicted": predicted_labels}
)
diagnosis_prediction_frame.to_csv(
    analysis_directory / "diagnosis_predictions.csv", index=False
)

Classes: ['ANCA Vasculitis', 'Acute COVID', 'Batch Control', 'Cardiac Sarcoidosis', 'Healthy', 'Hidradenitis Suppurativa', 'Long COVID', 'Lymphoma', 'MS', 'Melanoma', 'Multiple Myeloma', 'Premature Infant', 'Recovered COVID', 'Sepsis', 'Term Infant'], N=1823


  Fold 1: Accuracy = 0.820
  Fold 2: Accuracy = 0.878


  Fold 3: Accuracy = 0.774
  Fold 4: Accuracy = 0.904
  Fold 5: Accuracy = 0.888

Mean Accuracy: 0.853 +/- 0.049


In [14]:
confusion_counts = confusion_matrix(
    observed_labels, predicted_labels, labels=diagnosis_classes
)
normalized_confusion = confusion_counts.astype(float) / confusion_counts.sum(
    axis=1, keepdims=True
)

_, confusion_axis = plt.subplots(figsize=(11, 9))
sns.heatmap(
    normalized_confusion,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=diagnosis_classes,
    yticklabels=diagnosis_classes,
    ax=confusion_axis,
    vmin=0,
    vmax=1,
    square=True,
)
confusion_axis.set_xlabel("Predicted")
confusion_axis.set_ylabel("True")
confusion_axis.set_title("Diagnosis Classification (Normalized Confusion Matrix)")
plt.tight_layout()
plt.savefig(analysis_directory / "diagnosis_confusion_matrix.png", bbox_inches="tight")
plt.show()

---
## 7. Sex Prediction

Binary classification of sex from latent embeddings.

In [15]:
sex_frame = analysis_frame[analysis_frame["sex"].isin(["Male", "Female"])].reset_index(
    drop=True
)
sex_features = sex_frame[feature_columns].values
sex_labels = sex_frame["sex"].values
sex_groups = sex_frame["group_id"].values

stratified_folds = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=random_seed
)
observed_sex_labels, predicted_sex_labels = [], []
sex_accuracies = []

for training_indices, testing_indices in stratified_folds.split(
    sex_features, sex_labels, groups=sex_groups
):
    feature_scaler = StandardScaler()
    training_features = feature_scaler.fit_transform(sex_features[training_indices])
    testing_features = feature_scaler.transform(sex_features[testing_indices])

    classifier = LogisticRegression(max_iter=2_000, random_state=random_seed)
    classifier.fit(training_features, sex_labels[training_indices])
    fold_predictions = classifier.predict(testing_features)

    fold_accuracy = accuracy_score(sex_labels[testing_indices], fold_predictions)
    sex_accuracies.append(fold_accuracy)
    observed_sex_labels.extend(sex_labels[testing_indices])
    predicted_sex_labels.extend(fold_predictions)

mean_sex_accuracy = np.mean(sex_accuracies)
sex_accuracy_standard_deviation = np.std(sex_accuracies)
print(
    f"Sex Classification — Mean Accuracy: {mean_sex_accuracy:.3f} "
    f"+/- {sex_accuracy_standard_deviation:.3f}"
)

confusion_counts = confusion_matrix(
    observed_sex_labels,
    predicted_sex_labels,
    labels=["Female", "Male"],
)
normalized_confusion = confusion_counts.astype(float) / confusion_counts.sum(
    axis=1, keepdims=True
)

_, confusion_axis = plt.subplots(figsize=(4, 3))
sns.heatmap(
    normalized_confusion,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=["Female", "Male"],
    yticklabels=["Female", "Male"],
    ax=confusion_axis,
    vmin=0,
    vmax=1,
    square=True,
)
confusion_axis.set_xlabel("Predicted")
confusion_axis.set_ylabel("True")
confusion_axis.set_title("Sex Prediction")
plt.tight_layout()
plt.savefig(analysis_directory / "sex_confusion_matrix.png", bbox_inches="tight")
plt.show()

Sex Classification — Mean Accuracy: 0.681 +/- 0.009


---
## 8. Age Regression

Ridge regression to predict age from latent embeddings.

In [16]:
age_frame = analysis_frame[
    pd.to_numeric(analysis_frame["age"], errors="coerce").notna()
].copy()
age_frame["age"] = age_frame["age"].astype(float)

age_features = age_frame[feature_columns].values
age_labels = age_frame["age"].values
age_groups = age_frame["group_id"].values

folds = GroupKFold(n_splits=5)
observed_ages, predicted_ages = [], []

for training_indices, testing_indices in folds.split(
    age_features, age_labels, groups=age_groups
):
    feature_scaler = StandardScaler()
    training_features = feature_scaler.fit_transform(age_features[training_indices])
    testing_features = feature_scaler.transform(age_features[testing_indices])

    regressor = Ridge(alpha=1.0)
    regressor.fit(training_features, age_labels[training_indices])
    fold_predictions = regressor.predict(testing_features)

    observed_ages.extend(age_labels[testing_indices])
    predicted_ages.extend(fold_predictions)

observed_ages = np.array(observed_ages)
predicted_ages = np.array(predicted_ages)
mean_absolute_age_error = mean_absolute_error(observed_ages, predicted_ages)
age_r_squared = r2_score(observed_ages, predicted_ages)

print(
    f"Age Regression — MAE: {mean_absolute_age_error:.2f} years, "
    f"R2: {age_r_squared:.3f}"
)

_, regression_axis = plt.subplots(figsize=(5, 4))
regression_axis.scatter(
    observed_ages,
    predicted_ages,
    alpha=0.6,
    s=20,
    edgecolors="k",
    linewidths=0.5,
)
plot_limits = [
    min(observed_ages.min(), predicted_ages.min()) - 5,
    max(observed_ages.max(), predicted_ages.max()) + 5,
]
regression_axis.plot(
    plot_limits,
    plot_limits,
    "k--",
    alpha=0.5,
    label="Perfect prediction",
)
regression_axis.set_xlabel("True Age")
regression_axis.set_ylabel("Predicted Age")
regression_axis.set_title(
    f"Age Regression (MAE={mean_absolute_age_error:.1f}, R²={age_r_squared:.2f})"
)
regression_axis.legend(fontsize=8)
plt.tight_layout()
plt.savefig(analysis_directory / "age_regression.png", bbox_inches="tight")
plt.show()

Age Regression — MAE: 8.45 years, R2: 0.717


---
## 9. Cell Type Proportion Prediction

Predict cell type proportions (computed from the gated labels) using the latent embeddings via Ridge regression.

In [17]:
proportion_records = []
for sample_name, cell_type_tensor in zip(sample_names, sample_cell_types, strict=True):
    cell_type_names = dataset.get_cell_type_names(cell_type_tensor.numpy())
    cell_type_counts = Counter(cell_type_names)
    total_cells = sum(cell_type_counts.values())
    proportions = {
        cell_type: count / total_cells for cell_type, count in cell_type_counts.items()
    }
    proportions["sample_name"] = sample_name
    proportion_records.append(proportions)

proportion_frame = pd.DataFrame(proportion_records).fillna(0)
proportion_frame = proportion_frame.set_index("sample_name")

proportion_analysis_frame = pd.merge(
    embedding_frame,
    proportion_frame,
    left_on="sample_name",
    right_index=True,
    how="inner",
)
proportion_analysis_frame = proportion_analysis_frame.merge(
    matched_metadata[["sample_name", "group_id"]],
    on="sample_name",
    how="left",
)

cell_type_columns = [
    column
    for column in proportion_analysis_frame.columns
    if column not in feature_columns and column not in {"sample_name", "group_id"}
]
print(f"Cell types to predict: {cell_type_columns}")

Cell types to predict: ['Neutrophil', 'Monocyte Classical', 'NK cell', 'T cell CD4 Naive', 'T cell CD8 Mem', 'T cell CD8 Naive', 'T cell CD4 Mem', 'B cell', 'T cell gd', 'Basophil', 'Monocyte Nonclassical', 'T cell CD4 EMRA', 'Eosinophil', 'mDC', 'T cell DN', 'Plasmablast', 'pDC']


In [18]:
proportion_features = proportion_analysis_frame[feature_columns].values
proportion_groups = proportion_analysis_frame["group_id"].values
folds = GroupKFold(n_splits=5)

mean_absolute_errors = {}
for cell_type in cell_type_columns:
    observed_proportions = proportion_analysis_frame[cell_type].values
    fold_observations, fold_predictions = [], []

    for training_indices, testing_indices in folds.split(
        proportion_features, observed_proportions, groups=proportion_groups
    ):
        feature_scaler = StandardScaler()
        training_features = feature_scaler.fit_transform(
            proportion_features[training_indices]
        )
        testing_features = feature_scaler.transform(
            proportion_features[testing_indices]
        )

        regressor = Ridge(alpha=1.0)
        regressor.fit(training_features, observed_proportions[training_indices])
        fold_predictions.extend(regressor.predict(testing_features))
        fold_observations.extend(observed_proportions[testing_indices])

    mean_absolute_errors[cell_type] = mean_absolute_error(
        fold_observations, fold_predictions
    )

sorted_cell_types = sorted(mean_absolute_errors, key=mean_absolute_errors.__getitem__)
_, error_axis = plt.subplots(figsize=(8, 4))
error_axis.barh(
    range(len(sorted_cell_types)),
    [mean_absolute_errors[cell_type] for cell_type in sorted_cell_types],
    color="steelblue",
)
error_axis.set_yticks(range(len(sorted_cell_types)))
error_axis.set_yticklabels(sorted_cell_types, fontsize=8)
error_axis.set_xlabel("MAE (proportion)")
error_axis.set_title("Cell Type Proportion Prediction from Latent Embeddings")
plt.tight_layout()
plt.savefig(
    analysis_directory / "cell_type_proportion_prediction.png",
    bbox_inches="tight",
)
plt.show()

---
## 10. Marker-shift stress test

Measure embedding stability by applying random marker intensity shifts and comparing the resulting latent representations.

In [19]:
assert model is not None, "Model not loaded."


def apply_marker_shift(
    sample_tensor: torch.Tensor,
    scale_standard_deviation: float = 0.10,
    offset_standard_deviation: float = 0.05,
) -> torch.Tensor:
    """Apply one mild multiplicative/additive shift per marker."""
    _, _, marker_count = sample_tensor.shape
    scale = torch.exp(
        torch.randn(1, 1, marker_count, device=sample_tensor.device)
        * scale_standard_deviation
    ).to(sample_tensor.dtype)
    offset = (
        torch.randn(1, 1, marker_count, device=sample_tensor.device)
        * offset_standard_deviation
    ).to(sample_tensor.dtype)
    return torch.clamp(sample_tensor * scale + offset, min=0)


shifted_embeddings = {}
torch.manual_seed(random_seed)

for sample_index in tqdm(
    range(len(sample_tensors)), desc="Computing shifted embeddings"
):
    with torch.no_grad():
        sample_tensor = sample_tensors[sample_index].to(
            device=device, dtype=model_dtype
        )
        shifted_input = apply_marker_shift(sample_tensor)
        encoded_cells, *_ = model.student.forward_encoder(shifted_input)
        shifted_embedding, _, _ = model.student.forward_pooling(encoded_cells)
        shifted_embeddings[sample_names[sample_index]] = shifted_embedding.float().cpu()

sorted_sample_names = sorted(sample_embeddings)
embedding_distances = [
    np.linalg.norm(
        sample_embeddings[sample_name].flatten().numpy()
        - shifted_embeddings[sample_name].flatten().numpy()
    )
    for sample_name in sorted_sample_names
]
pd.DataFrame(
    {"sample_name": sorted_sample_names, "embedding_l2_distance": embedding_distances}
).to_csv(analysis_directory / "marker_shift_distances.csv", index=False)

print(f"Mean L2 distance (original vs shifted): {np.mean(embedding_distances):.4f}")
print(f"Standard deviation of L2 distance: {np.std(embedding_distances):.4f}")

_, distance_axis = plt.subplots(figsize=(5, 3))
distance_axis.hist(embedding_distances, bins=20, color="steelblue", edgecolor="white")
distance_axis.axvline(
    np.mean(embedding_distances),
    color="red",
    linestyle="--",
    label=f"Mean = {np.mean(embedding_distances):.3f}",
)
distance_axis.set_xlabel("L2 Distance")
distance_axis.set_ylabel("Count")
distance_axis.set_title("Embedding Stability Under Marker Shifts")
distance_axis.legend(fontsize=8)
plt.tight_layout()
plt.savefig(analysis_directory / "marker_shift_stability.png", bbox_inches="tight")
plt.show()

Computing shifted embeddings:   0%|          | 0/1870 [00:00<?, ?it/s]

Mean L2 distance (original vs shifted): 2.6256
Standard deviation of L2 distance: 1.2196


---
## 11. Pool Attention Interpretation

Analyze which cell types receive the most attention during pooling, revealing what the model considers most informative.

In [20]:
assert model is not None, "Model not loaded."

attention_sample_names = []
attention_results = []

for sample_index in tqdm(range(len(sample_tensors)), desc="Extracting pool attention"):
    with torch.no_grad():
        sample_tensor = sample_tensors[sample_index].to(
            device=device, dtype=model_dtype
        )
        cell_types = sample_cell_types[sample_index]

        encoded_cells, *_ = model.student.forward_encoder(sample_tensor)
        _, _, pooling_attention = model.student.forward_pooling(encoded_cells)

        # Pooling attention has shape [heads, seeds, cells].
        attention_weights = (
            pooling_attention.mean(dim=0).squeeze(0).float().cpu().numpy()
        )

        cell_type_names = dataset.get_cell_type_names(cell_types)

        attention_by_cell_type = {}
        for cell_type, attention_weight in zip(
            cell_type_names, attention_weights, strict=True
        ):
            if cell_type not in attention_by_cell_type:
                attention_by_cell_type[cell_type] = []
            attention_by_cell_type[cell_type].append(attention_weight)

        mean_attention_by_cell_type = {
            cell_type: np.mean(weights)
            for cell_type, weights in attention_by_cell_type.items()
        }
        attention_results.append(mean_attention_by_cell_type)
        attention_sample_names.append(sample_names[sample_index])

all_cell_types = sorted(
    {
        cell_type
        for attention_result in attention_results
        for cell_type in attention_result
    }
)
attention_records = []
for sample_index, sample_name in enumerate(attention_sample_names):
    record = {"sample_name": sample_name}
    for cell_type in all_cell_types:
        record[cell_type] = attention_results[sample_index].get(cell_type, 0.0)
    attention_records.append(record)

attention_frame = pd.DataFrame(attention_records)
attention_frame.to_csv(analysis_directory / "pool_attention_by_sample.csv", index=False)
print(f"Attention DataFrame shape: {attention_frame.shape}")
attention_frame.head()

Extracting pool attention:   0%|          | 0/1870 [00:00<?, ?it/s]

Attention DataFrame shape: (1870, 18)


,sample_name,B cell,Basophil,Eosinophil,Monocyte Classical,Monocyte Nonclassical,NK cell,Neutrophil,Plasmablast,T cell CD4 EMRA,T cell CD4 Mem,T cell CD4 Naive,T cell CD8 Mem,T cell CD8 Naive,T cell DN,T cell gd,mDC,pDC
0,Heparin MDIPA_DORA_HD_05Nov24,0.000015,0.000004,0.000007,0.000012,0.000008,0.000009,0.000008,0.000018,0.000011,0.000011,0.000023,0.000011,0.000023,0.000013,0.000011,0.000013,0.000014
1,MDIPA_845061_097_Blood Draw_PBMC,0.000009,0.000012,0.000009,0.000012,0.000008,0.000007,0.000010,0.000012,0.000010,0.000009,0.000007,0.000017,0.000010,0.000010,0.000009,0.000013,0.000013
2,MDIPA_845061_097_Blood Draw_WBHep,0.000009,0.000009,0.000009,0.000016,0.000010,0.000010,0.000009,0.000036,0.000014,0.000009,0.000007,0.000013,0.000014,0.000016,0.000014,0.000024,0.000038
3,MDIPA_845061_098_Blood Draw_PBMC,0.000008,0.000009,0.000008,0.000009,0.000010,0.000009,0.000009,0.000008,0.000010,0.000011,0.000009,0.000010,0.000009,0.000011,0.000009,0.000010,0.000008
4,MDIPA_845061_098_Blood Draw_WBHep,0.000015,0.000015,0.000019,0.000020,0.000013,0.000010,0.000010,0.000017,0.000007,0.000007,0.000005,0.000006,0.000009,0.000013,0.000011,0.000017,0.000019


In [21]:
mean_attention = attention_frame[all_cell_types].mean().sort_values(ascending=True)

_, attention_axis = plt.subplots(figsize=(7, 4))
bar_colors = [
    cell_type_colors.get(cell_type, default_color) for cell_type in mean_attention.index
]
attention_axis.barh(range(len(mean_attention)), mean_attention.values, color=bar_colors)
attention_axis.set_yticks(range(len(mean_attention)))
attention_axis.set_yticklabels(mean_attention.index, fontsize=8)
attention_axis.set_xlabel("Mean Pool Attention Weight")
attention_axis.set_title("Pool Attention by Cell Type")
plt.tight_layout()
plt.savefig(analysis_directory / "pool_attention_by_cell_type.png", bbox_inches="tight")
plt.show()

In [22]:
attention_with_metadata = pd.merge(
    attention_frame, matched_metadata, on="sample_name", how="inner"
)

attention_diagnoses = display_diagnoses[:6]
_, subplot_axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)

for subplot_axis, diagnosis in zip(
    subplot_axes.ravel(), attention_diagnoses, strict=True
):
    diagnosis_attention = attention_with_metadata[
        attention_with_metadata["analysis_diagnosis"] == diagnosis
    ]
    mean_values = diagnosis_attention[all_cell_types].mean().sort_values(ascending=True)
    bar_colors = [
        cell_type_colors.get(cell_type, default_color)
        for cell_type in mean_values.index
    ]
    subplot_axis.barh(range(len(mean_values)), mean_values.values, color=bar_colors)
    subplot_axis.set_yticks(range(len(mean_values)))
    subplot_axis.set_yticklabels(mean_values.index, fontsize=7)
    subplot_axis.set_title(f"{diagnosis} (n={len(diagnosis_attention)})", fontsize=10)
    subplot_axis.set_xlabel("Mean Attention")

plt.suptitle("Pool Attention Stratified by Diagnosis", fontsize=13)
plt.tight_layout()
plt.savefig(analysis_directory / "pool_attention_by_diagnosis.png", bbox_inches="tight")
plt.show()

---
## Summary

This notebook demonstrated the key downstream analyses enabled by MAESTRO:

- **Reconstruction**: The model can reconstruct masked cells, preserving cell type structure.
- **Latent embeddings**: Each patient is compressed to a fixed-size vector capturing immune state.
- **Classification/Regression**: Latent embeddings support diagnosis, sex, and age prediction.
- **Cell type proportions**: The model implicitly learns cell type composition.
- **Batch robustness**: Embeddings are stable under simulated batch effects.
- **Attention**: Pool attention reveals which cell types are most informative.

Primary reusable outputs are written to the run's `analysis/` directory. Start with `maestro_embeddings_with_metadata.csv` for downstream cohort-level analyses and use `maestro_embeddings.csv` when sharing embeddings without clinical metadata. Attention is associative rather than causal, and the grouped probes above remain exploratory because pretraining saw the full corpus.